# Lecture 02 — Multiple Linear Regression for Predictive Analytics (Python)

**Term:** Fall 2025  
**Week/Topic:** Lecture 02 — Multiple Linear Regression (MLR)  
**Instructor:** Dr. Bushaj  

### What we'll cover
- Quick recap: simple vs. multiple linear regression; outcome, predictors, coefficients, error
- Dataset: **Toyota Corolla used-car pricing** (features like Age, KM, HP, Fuel Type, Doors, Weight, etc.)
- Preprocessing for MLR
  - Clean column names; select/rename features
  - Handle missing values & outliers; transform skewed targets (optional log)
  - Encode categoricals (one-hot/dummies): `Fuel_Type`, `Automatic`, `Met_Color`
  - Train/validation split (and optional test) with reproducible seeds
  - Optional scaling (when useful for regularization)
- Fit baseline OLS models
  - `sklearn.linear_model.LinearRegression` (predict/score)
  - `statsmodels.api.OLS` for rich summaries (coeffs, CIs, p-values, F-test, R²/Adj. R²)
  - Interpret coefficients (holding others constant); practical effect sizes
- Diagnose assumptions & multicollinearity
  - Residual plots vs. fitted; QQ-plot; homoscedasticity checks
  - Correlations & **VIF** to detect multicollinearity
- Evaluate predictive accuracy (train vs. validation)
  - `MAE`, `RMSE`, `R²`, `MAPE`, and error direction (`ME`, `MPE`)
  - Compare baseline vs. improved models; avoid overfitting
- Model/feature selection
  - Forward / backward / stepwise (concepts & simple programmatic demos)
  - Criteria: **Adjusted R²**, **AIC**, **BIC**
- Regularization for improved generalization
  - **Ridge** and **Lasso** with cross-validation (`RidgeCV`, `LassoCV`)
  - Interpret shrinkage paths; discuss Elastic Net option
- Final model & use
  - Pick a parsimonious model; save pipeline; predict on new records

---

## Import Required Libraries

In [ ]:
%matplotlib inline

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, BayesianRidge
import statsmodels.formula.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pylab as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score

In [ ]:
!pip install dmba

from dmba import regressionSummary, exhaustive_search
from dmba import backward_elimination, forward_selection, stepwise_selection
from dmba import adjusted_r2_score, AIC_score, BIC_score

## Load the Data

In [ ]:
#create a folder with all the data files there and get the path to that folder
my_drive_path = "/content/drive/MyDrive/SUNY/Class Material/2024 Fall/MSA550A/Python Class Work/msa550-code-files/data/"

In [ ]:
car_df = pd.read_csv(my_drive_path+'ToyotaCorolla.csv')
print(car_df.head(5))
car_df.columns

| Variable           | Description                                                   |
| ------------------ | ------------------------------------------------------------- |
| **Price**          | Price of the car in Euros (dependent variable for regression) |
| **Age\_08\_04**    | Age of the car (in months, as of August 2004)                 |
| **KM**             | Kilometers driven                                             |
| **Fuel\_Type**     | Fuel type (Diesel, Petrol, CNG)                               |
| **HP**             | Horsepower                                                    |
| **Met\_Color**     | Metallic color (1 = Yes, 0 = No)                              |
| **Automatic**      | Automatic transmission (1 = Yes, 0 = No)                      |
| **CC**             | Engine cylinder capacity (cc)                                 |
| **Doors**          | Number of doors                                               |
| **Quarterly\_Tax** | Road tax in Euros (quarterly)                                 |
| **Weight**         | Weight of the car (kg)                                        |


## Preprocessing the Data

### Handling Missing Values
Remember, the fact that it has no null values, does not mean no missing values.

In [ ]:
print("Missing values:\n", car_df.isnull().sum())

### Correlation Table
Multicollinearity check, variable relationship etc.

In [ ]:
numeric_columns = car_df.select_dtypes(include=[np.number]).columns

# Create a new DataFrame with only numeric columns
numeric_df = car_df[numeric_columns]

correlation_matrix = numeric_df.corr()
#print(correlation_matrix)

fig = go.Figure(data=go.Heatmap(
                z=correlation_matrix.values,
                x=correlation_matrix.columns,
                y=correlation_matrix.index,
                colorscale='RdBu',
                zmin=-1, zmax=1,
                text=correlation_matrix.values,
                texttemplate='%{text:.2f}',
                hoverongaps=False))

# Update the layout
fig.update_layout(
    title='Correlation Matrix',
    xaxis_title='Features',
    yaxis_title='Features',
    width=800,  # Adjust the width as needed
    height=700  # Adjust the height as needed
)

# Show the plot
fig.show()

1. Strong correlations with Price:
* Age_08_04 has a strong negative correlation (-0.876590) with Price, indicating that older cars tend to be cheaper.
* Mfg_Year has a strong positive correlation (0.885159) with Price, which is consistent with the Age correlation.
* KM (kilometers) has a moderate negative correlation (-0.569960) with Price, suggesting that cars with higher mileage tend to be cheaper.

2. Multicollinearity:
* Age_08_04 and Mfg_Year are highly correlated (-0.983661), which is expected as they represent similar information. **drop Mfg_Year**
* Id seems to be correlated with several variables, which is unusual and might indicate it's not just a simple identifier. **drop Id**

3. Lack of correlation:
* Some features like Radio, Power_Steering, and Metallic_Rim show very weak correlations with Price. **will drop them**

4. Potential redundant features:
* Radio and Radio_cassette are highly correlated (0.991621), suggesting they might be redundant. *drop them - (radio was also mentioned in point 3)**

In [ ]:
# for Radio_cassette, Radio, Power_Steering, and Metallic_Rim, Mfg_Year Id we planned to drop them already

#in addition: CD_Player, Backseat_Divider, BOVAG_Guarantee, Guarantee_Period, ABS, Automatic_airco

columns_to_drop = ['Radio_cassette', 'Radio', 'Power_Steering', 'Metallic_Rim', 'Mfg_Year', 'Id','CD_Player','Backseat_Divider','BOVAG_Guarantee','Guarantee_Period','Automatic_airco','ABS']

car_df = car_df.drop(columns=columns_to_drop)

print("Columns dropped successfully.")
print(f"Remaining columns: {car_df.columns.tolist()}")

### Check for Outliers

In [ ]:
# 4. Check for outliers
numeric_columns = car_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create subplots, one for each numeric column
fig = make_subplots(rows=1, cols=len(numeric_columns),
                    subplot_titles=numeric_columns)

# Add box plots for each numeric column
for i, col in enumerate(numeric_columns):
    fig.add_trace(
        go.Box(y=car_df[col], name=col),
        row=1, col=i+1
    )

# Update layout
fig.update_layout(
    title_text="Boxplots to Check for Outliers",
    height=600,
    width=200 * len(numeric_columns),  # Adjust width based on number of plots
    showlegend=False
)

# Update x-axis
fig.update_xaxes(visible=False)  # Hide x-axis labels as they're not needed for boxplots

# Show plot
fig.show()

In [ ]:
#outlier stats
numeric_columns = car_df.select_dtypes(include=['int64', 'float64']).columns

outlier_percentages = {}

# Print summary statistics for each numeric column
for column in numeric_columns:
    print(f"\nSummary statistics for {column}:")
    print(car_df[column].describe())
    print("\nPotential outliers:")
    Q1 = car_df[column].quantile(0.25)
    Q3 = car_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = car_df[(car_df[column] < lower_bound) | (car_df[column] > upper_bound)][column]
    print(outliers)

    # Calculate percentage of outliers
    outlier_percentage = (len(outliers) / len(car_df)) * 100
    outlier_percentages[column] = outlier_percentage

    print(f"Percentage of outliers: {outlier_percentage:.2f}%")
    print("--------------------")

# Print total percentage of outliers for each parameter
print("\nTotal percentage of outliers for each parameter:")
for column, percentage in outlier_percentages.items():
    print(f"{column}: {percentage:.2f}%")

In [ ]:
def remove_outliers(df, columns):
    df_clean = df.copy()
    total_removed = 0
    total_data = len(df) * len(columns)

    for column in columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
        df_clean = df_clean[~df_clean.index.isin(outliers.index)]

        num_removed = len(outliers)
        total_removed += num_removed

        print(f"Column {column}:")
        print(f"  Number of outliers removed: {num_removed}")
        print(f"  Percentage of data removed: {(num_removed / len(df)) * 100:.2f}%")
        print("--------------------")

    overall_percentage = (total_removed / total_data) * 100
    print(f"\nOverall percentage of data points removed: {overall_percentage:.2f}%")
    print(f"Number of rows in original dataset: {len(df)}")
    print(f"Number of rows in cleaned dataset: {len(df_clean)}")

    return df_clean

print(car_df.shape)
# Remove outliers - be careful with this one!!!
car_df_clean = remove_outliers(car_df, numeric_columns)
car_df_clean.shape

### Encode Categorical Variables

In [ ]:
# 5. Encode categorical variables
car_df_model = pd.get_dummies(car_df, columns=['Fuel_Type'], drop_first=True)

In [ ]:
car_df_model.columns

### Scale Numerical Variables


In [ ]:
scaler = StandardScaler()
#hand picked them
numerical_columns = ['Age_08_04', 'KM', 'HP', 'CC', 'Weight']
car_df_model[numerical_columns] = scaler.fit_transform(car_df[numerical_columns])

### Feature Engineering and Transformations
You can create new features to better represent data.
You can transform variables as well

In [ ]:
# Feature engineering (example)  -- It captures the relationship between price and mileage in a single feature
car_df['Price_per_KM'] = car_df['Price'] / (car_df['KM'] + 1)  # Adding 1 to avoid division by zero


# Logarithmic transformation of Price - It can help normalize the distribution of prices if they are right-skewed
car_df['Log_Price'] = np.log(car_df['Price']) #In economic models, log-transformations are often used because many economic phenomena follow log-normal distributions.


In [ ]:
# Additional: Check VIF for multicollinearity
def calculate_vif(X):
    vif = pd.DataFrame()
    vif["Variable"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif

print("VIF:\n", calculate_vif(numeric_df))

## Split the Data

In [ ]:
#intentionally selecting a subset of the features

# 'Age_08_04', 'KM', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'HP', 'Met_Color', 'Automatic', 'CC', 'Doors', 'Weight'

predictors = ['Age_08_04', 'KM', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'HP', 'Met_Color', 'Automatic', 'CC', 'Doors', 'Quarterly_Tax', 'Weight']
outcome = 'Price'

#select only top 1000 records
#car_df_model = car_df_model.head(1000)



X = car_df_model[predictors]
y = car_df_model[outcome]





train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.3, random_state=42)